# Modulo 1 · Notebook 1 — Teoria de Feature Engineering

El **feature engineering** es el arte de convertir datos crudos en entradas
numericas (las "features") con las que un modelo de machine learning puede
aprender bien. Una buena analogia: el modelo es un cocinero y las features son
los ingredientes ya picados y medidos. Por muy buen cocinero que sea, si le das
ingredientes en mal estado el plato no saldra bien.

Un modelo solo es tan bueno como la **representacion** que recibe. Las
transformaciones correctas logran tres cosas:

- hacen que los patrones sean *separables* (lineal o localmente),
- estabilizan la optimizacion (que el entrenamiento converja),
- y codifican conocimiento del dominio que el modelo, de otro modo, tendria que
  inferir desde cero.

En este notebook recorremos las transformaciones mas usadas. Para cada una vemos
primero **(a)** la intuicion y luego **(b)** un ejemplo ejecutable sobre el
dataset real del **Titanic**:

1. One-hot encoding
2. Feature hashing (el "hashing trick")
3. Bucketing / binning
4. Estandarizacion (z-score)
5. Normalizacion (min-max y L2)
6. Reduccion de dimensionalidad (PCA)
7. Embeddings (vectores densos aprendidos)

Cerramos con una **chuleta** que resume cuando usar cada tecnica.


## Setup y datos

Usamos el dataset del Titanic que viene con seaborn. Si estas sin conexion,
`seaborn.load_dataset` no podra descargarlo, asi que lo envolvemos en un
`try/except` y caemos a un pequeno dataframe sintetico con las mismas columnas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)

def load_titanic():
    try:
        import seaborn as sns
        df = sns.load_dataset("titanic")
        if df is None or len(df) == 0:
            raise ValueError("vacio")
        print("Dataset real del Titanic cargado desde seaborn.")
        return df
    except Exception as e:  # sin conexion / sin seaborn-data
        print(f"Usando datos sinteticos ({type(e).__name__}: {e})")
        n = 500
        df = pd.DataFrame({
            "survived": np.random.randint(0, 2, n),
            "pclass": np.random.choice([1, 2, 3], n, p=[0.2, 0.2, 0.6]),
            "sex": np.random.choice(["male", "female"], n, p=[0.65, 0.35]),
            "age": np.clip(np.random.normal(29, 14, n), 0.4, 80).round(1),
            "sibsp": np.random.poisson(0.5, n),
            "parch": np.random.poisson(0.4, n),
            "fare": np.round(np.random.exponential(32, n), 4),
            "embarked": np.random.choice(["S", "C", "Q"], n, p=[0.72, 0.19, 0.09]),
            "class": np.random.choice(["First", "Second", "Third"], n),
            "embark_town": np.random.choice(
                ["Southampton", "Cherbourg", "Queenstown"], n, p=[0.72, 0.19, 0.09]),
        })
        return df

df = load_titanic()
print(df.shape)
df.head()

In [ ]:
# Un vistazo rapido a tipos y valores faltantes: las decisiones de ingenieria
# dependen de esto.
df.info()
df.isna().sum()

## 1. One-hot encoding

**Intuicion primero.** Imagina la columna `embarked` con tres puertos: S, C, Q.
Si los codificas como 0, 1, 2 le estas mintiendo al modelo: le dices que Q (2)
esta "el doble de lejos" de S (0) que C (1), y que hay un orden. Pero entre
puertos no hay orden ni distancia natural. **One-hot** resuelve esto dandole a
cada categoria su propia columna indicadora (0/1), de modo que todas las
categorias quedan a la misma distancia entre si.

**La formula, en compacto.** Una variable categorica con $K$ niveles se mapea a
un vector con un solo 1:

$$\text{onehot}(c) = e_c \in \{0,1\}^{K}, \qquad (e_c)_j = 1 \text{ si } j=c, \text{ si no } 0.$$

Exactamente una componente vale 1, asi que los vectores son ortogonales y
equidistantes: no hay orden espurio.

**Cuidado con la cardinalidad.** El encoding agrega $K$ columnas por feature (o
$K-1$ si *eliminas una* para evitar la trampa de las variables dummy /
colinealidad). Para features de alta cardinalidad (IDs de usuario, codigos
postales) $K$ explota y produce matrices enormes y **dispersas** (casi todo
ceros). Esa es justo la motivacion del *hashing trick* (seccion 2) y de los
*embeddings* (seccion 7).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["sex", "embarked"]
work = df.dropna(subset=cat_cols).copy()

# Version comoda con pandas (drop_first evita la trampa de las dummy).
dummies = pd.get_dummies(work[cat_cols], drop_first=False)
print("Columnas de pd.get_dummies:", list(dummies.columns))
dummies.head()

In [ ]:
# Version sklearn: el camino de produccion (ajusta en train, transforma en test).
enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X = enc.fit_transform(work[cat_cols])
print("Categorias:", enc.categories_)
print("Forma codificada:", X.shape, "| dtype:", X.dtype, "| guardado como matriz dispersa")
print(f"Densidad: {X.nnz / (X.shape[0]*X.shape[1]):.3f} "
      f"(fraccion de entradas distintas de cero)")
X[:5].toarray()

## 2. Feature hashing (el "hashing trick")

**Intuicion primero.** One-hot tiene dos problemas: necesita *conocer todas las
categorias de antemano* y crece con la cardinalidad. El hashing es como tener un
archivero con un numero **fijo** de cajones: tomas el nombre de cada categoria,
le aplicas una funcion hash y eso te dice en que cajon va. No importa cuantas
categorias aparezcan: el numero de cajones $m$ **lo eliges tu**.

**La formula, en compacto.** Cada feature (nombre=valor) se manda a uno de $m$
buckets mediante un hash $h$:

$$\phi(x)_j = \sum_{i\,:\,h(i)=j} \xi(i)\, x_i, \qquad j = 0,\dots,m-1,$$

donde $\xi(i)\in\{-1,+1\}$ es un hash de signo que mantiene el mapeo
(aproximadamente) insesgado.

**Colisiones.** Como $m$ es fijo, dos categorias distintas pueden caer en el
mismo cajon (una colision). Con $d$ features activas y $m$ buckets, la
probabilidad de que un par colisione es $\approx 1/m$; elegir $m \gg d$ las hace
raras. El hash de signo $\xi$ hace que las contribuciones que colisionan se
cancelen en promedio en vez de sumarse siempre.

**El trade-off.** El hashing es *sin estado*: no hay vocabulario que guardar,
maneja categorias nuevas gratis y acota la memoria a $m$. El costo es una pequena
perdida de precision (por colisiones) y de interpretabilidad (no puedes mapear un
bucket de vuelta a su categoria).

In [ ]:
from sklearn.feature_extraction import FeatureHasher

# Hashea una columna categorica a un numero FIJO y pequeno de buckets.
n_buckets = 8
hasher = FeatureHasher(n_features=n_buckets, input_type="string")

tokens = [[f"embark={v}"] for v in work["embarked"].astype(str)]
H = hasher.transform(tokens)
print(f"{work['embarked'].nunique()} categorias -> {n_buckets} buckets fijos")
print("Forma hasheada:", H.shape)
pd.DataFrame(H.toarray()[:6],
             index=work["embarked"].head(6).values).round(0)

In [ ]:
# Demostramos colisiones: achicamos el espacio de buckets por debajo de la cardinalidad.
def bucket_of(value, m):
    h = FeatureHasher(n_features=m, input_type="string")
    row = h.transform([[f"town={value}"]]).toarray()[0]
    return int(np.flatnonzero(row)[0]) if row.any() else None

towns = work["embark_town"].dropna().unique()
for m in (16, 2):
    mapping = {t: bucket_of(t, m) for t in towns}
    n_collide = len(mapping) - len(set(mapping.values()))
    print(f"m={m:>2} buckets -> {mapping}  | colisiones: {n_collide}")

## 3. Bucketing / binning (discretizacion)

**Intuicion primero.** A veces lo que importa no es el valor exacto sino el
*tramo*: para sobrevivir en el Titanic quiza no importa si tienes 4 o 6 anos,
sino si eres "nino" o "adulto". El binning convierte una variable continua en
cajones ordenados, lo que ayuda a modelos no lineales, domestica outliers y
captura efectos de umbral.

Dos estrategias comunes:

- **Uniforme (igual ancho):** parte el rango $[\min, \max]$ en $k$ intervalos del
  mismo ancho $w = \frac{\max-\min}{k}$. Simple, pero sensible a sesgo/outliers
  (algunos cajones quedan casi vacios).
- **Por cuantiles (igual frecuencia):** pone los bordes en los cuantiles
  $\tfrac{i}{k}$ para que cada cajon tenga ~$n/k$ muestras. Robusto al sesgo; los
  anchos varian.

Columnas de dinero/edad sesgadas (como `fare`) suelen quedar mejor por
**cuantiles**.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

age = df["age"].dropna().to_frame()

# pandas: uniforme (cut) vs cuantiles (qcut)
uniform = pd.cut(age["age"], bins=4)
quantile = pd.qcut(age["age"], q=4)
print("Conteos por bin uniforme (igual ancho):")
print(uniform.value_counts().sort_index())
print("\nConteos por bin de cuantiles (igual frecuencia):")
print(quantile.value_counts().sort_index())

In [ ]:
# sklearn KBinsDiscretizer: cajones codificados como ordinales, listos para un modelo.
kb_uniform = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
kb_quant   = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")

age_u = kb_uniform.fit_transform(age).ravel()
age_q = kb_quant.fit_transform(age).ravel()
print("Bordes uniformes:  ", np.round(kb_uniform.bin_edges_[0], 1))
print("Bordes por cuantil:", np.round(kb_quant.bin_edges_[0], 1))

fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
ax[0].hist(age_u, bins=np.arange(-0.5, 4.5), rwidth=0.8); ax[0].set_title("Uniforme")
ax[1].hist(age_q, bins=np.arange(-0.5, 4.5), rwidth=0.8); ax[1].set_title("Cuantiles")
for a in ax: a.set_xlabel("bin"); a.set_xticks(range(4))
ax[0].set_ylabel("conteo"); plt.tight_layout(); plt.show()

## 4. Estandarizacion (z-score)

**Intuicion primero.** Muchos algoritmos (regresion lineal/logistica con
regularizacion, SVMs, k-NN, k-means, PCA, redes neuronales) miden *distancias* o
calculan *gradientes*. Si una feature esta en metros y otra en milimetros, la de
milimetros domina solo por estar en una unidad mas grande, no porque importe
mas. La estandarizacion pone a todas las features en la misma escala: media 0 y
varianza 1.

**La formula, en compacto.**

$$z = \frac{x - \mu}{\sigma}, \qquad \mu = \tfrac{1}{n}\sum_i x_i, \quad \sigma = \sqrt{\tfrac{1}{n}\sum_i (x_i-\mu)^2}.$$

Despues de esto cada feature tiene $\mathbb{E}[z]=0$ y $\operatorname{Var}[z]=1$,
asi que ninguna domina una distancia o un gradiente solo por sus unidades.

**Ajusta solo en train.** Calcula $\mu,\sigma$ en el set de entrenamiento y
reutilizalos en validacion/test para evitar fugas de informacion (*leakage*).

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["age", "fare"]
num = df[num_cols].dropna()

scaler = StandardScaler().fit(num)
z = scaler.transform(num)
print("Medias aprendidas:", scaler.mean_.round(3))
print("Desv. estandar   :", np.sqrt(scaler.var_).round(3))
print("Tras escalar -> media:", z.mean(axis=0).round(3),
      " std:", z.std(axis=0).round(3))

# chequeo manual de la formula del z-score para el primer valor de 'age'
x0 = num["age"].iloc[0]
print(f"\nz manual para age={x0}: "
      f"{(x0 - scaler.mean_[0]) / np.sqrt(scaler.var_[0]):.4f}  "
      f"== sklearn: {z[0,0]:.4f}")

## 5. Normalizacion (min-max y L2)

"Normalizacion" es un termino sobrecargado: nombra dos transformaciones
distintas.

**Min-max** lleva cada feature a un rango fijo, normalmente $[0,1]$. Intuicion:
es como un termometro reescalado donde el minimo es 0 y el maximo es 1.

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}.$$

Util cuando necesitas entradas acotadas (pixeles de imagen, algunas redes) y la
distribucion no tiene colas pesadas. Es **sensible a outliers**: un valor extremo
estira el rango y comprime a todos los demas.

**Normalizacion L2 (vectorial)** reescala cada *muestra* (fila) a longitud 1.
Intuicion: te quedas solo con la *direccion* del vector, no con su magnitud.

$$\mathbf{x}' = \frac{\mathbf{x}}{\lVert \mathbf{x}\rVert_2}, \qquad \lVert \mathbf{x}\rVert_2 = \sqrt{\textstyle\sum_j x_j^2}.$$

Ahora $\lVert \mathbf{x}'\rVert_2 = 1$. Es comun en texto/TF-IDF y donde se usa
similitud coseno. Ojo a la diferencia: StandardScaler/MinMax actuan **por
columna**; Normalizer actua **por fila**.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, Normalizer

mm = MinMaxScaler().fit(num)
num_mm = mm.transform(num)
print("min-max -> min:", num_mm.min(axis=0).round(3),
      " max:", num_mm.max(axis=0).round(3))

# Normaliza L2 cada FILA (muestra) de los vectores de dos features.
l2 = Normalizer(norm="l2")
num_l2 = l2.fit_transform(num.values[:5])
print("\nPrimeras 5 filas tras normalizacion L2:")
print(np.round(num_l2, 3))
print("Normas de fila (deben dar 1):", np.round(np.linalg.norm(num_l2, axis=1), 3))

## 6. Reduccion de dimensionalidad — PCA

**Intuicion primero.** Imagina que fotografias una nube de puntos 3D. Quieres
elegir el angulo de la camara que muestre la *mayor variacion* posible: ese
angulo "ve" mas estructura. PCA hace exactamente eso pero en cualquier numero de
dimensiones: encuentra direcciones (las **componentes principales**) ordenadas
por cuanta varianza capturan, y te quedas con las primeras.

**La formula, en compacto.** Con datos estandarizados $X \in \mathbb{R}^{n\times d}$
se forma la matriz de covarianza

$$\Sigma = \frac{1}{n-1} X^\top X,$$

y se hace su descomposicion en autovalores: $\Sigma v_k = \lambda_k v_k$. Los
autovectores $v_k$ son las direcciones (ortonormales); el autovalor $\lambda_k$ es
la varianza a lo largo de $v_k$. La **proporcion de varianza explicada** por la
componente $k$ es

$$\text{EVR}_k = \frac{\lambda_k}{\sum_j \lambda_j}.$$

Proyectando sobre las $r$ primeras componentes, $Z = X V_r$, obtienes una
representacion mas compacta que retiene la mayor varianza posible.
**Estandariza primero**, o las features de mayor varianza (unidades grandes)
dominaran las componentes.

In [ ]:
from sklearn.decomposition import PCA

pca_cols = ["age", "fare", "sibsp", "parch", "pclass"]
Xp = df[pca_cols].dropna()
Xp_std = StandardScaler().fit_transform(Xp)

pca = PCA().fit(Xp_std)
evr = pca.explained_variance_ratio_
print("Varianza explicada por componente:", evr.round(3))
print("Acumulada:", np.cumsum(evr).round(3))

# Chequeo: los autovalores de PCA == autovalores de la matriz de covarianza.
cov = np.cov(Xp_std, rowvar=False)
eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
print("\nPCA explained_variance_:", pca.explained_variance_.round(3))
print("Autovalores np.linalg  :", eigvals.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) scree / varianza explicada acumulada
ax[0].bar(range(1, len(evr)+1), evr, alpha=0.6, label="individual")
ax[0].plot(range(1, len(evr)+1), np.cumsum(evr), "o-r", label="acumulada")
ax[0].axhline(0.9, ls="--", c="gray"); ax[0].set_xlabel("componente")
ax[0].set_ylabel("varianza explicada"); ax[0].set_title("Scree plot"); ax[0].legend()

# (b) datos proyectados sobre las dos primeras componentes principales
Z = PCA(n_components=2).fit_transform(Xp_std)
target = df.loc[Xp.index, "survived"]
sc = ax[1].scatter(Z[:, 0], Z[:, 1], c=target, cmap="coolwarm", s=12, alpha=0.6)
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
ax[1].set_title("Titanic proyectado sobre PC1/PC2"); plt.colorbar(sc, ax=ax[1], label="survived")
plt.tight_layout(); plt.show()

## 7. Embeddings — vectores densos aprendidos

**Intuicion primero.** Los vectores one-hot son **dispersos, enormes y
equidistantes**: cada categoria esta a la misma distancia de todas las demas, asi
que no codifican *parecido*. Un **embedding** le asigna a cada categoria un vector
denso *aprendido* $\mathbf{e}_c \in \mathbb{R}^{d}$ con $d \ll K$. La magia: como
ese vector se aprende junto con el modelo, categorias parecidas terminan *cerca*
en el espacio (piensa en "rey" y "reina" quedando cerca en NLP).

**La formula, en compacto.**

$$c \;\longmapsto\; \mathbf{e}_c = E^\top\, \text{onehot}(c), \qquad E \in \mathbb{R}^{K\times d}.$$

La matriz $E$ (la *tabla de embeddings*) se **entrena por descenso de gradiente
junto con el modelo**. Mecanicamente, buscar un embedding es solo *seleccionar
una fila* de $E$: equivale a multiplicar el one-hot por $E$, pero se hace como un
indice $O(1)$ en lugar de una multiplicacion de matrices dispersa.

Los embeddings son la forma estandar de manejar categoricas de alta cardinalidad
(usuarios, productos, palabras) y son la base del NLP moderno y de los sistemas
de recomendacion.

In [ ]:
# Ejemplo minimo de nn.Embedding (PyTorch) para la categorica 'embark_town'.
# (Cae a una ilustracion con NumPy si torch no esta disponible.)
sex_cat = df["embark_town"].dropna().astype("category")
vocab = list(sex_cat.cat.categories)
idx = sex_cat.cat.codes.to_numpy()
print("Vocabulario:", vocab)

try:
    import torch
    import torch.nn as nn
    torch.manual_seed(0)

    embed_dim = 3
    emb = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim)
    sample = torch.tensor(idx[:5])
    vectors = emb(sample)
    print(f"\nnn.Embedding: {len(vocab)} categorias -> vectores densos de {embed_dim}-d")
    print("Tabla de embeddings E (una fila por categoria):")
    print(emb.weight.detach().numpy().round(3))
    print("\nBusqueda para las primeras 5 filas (indices", idx[:5], "):")
    print(vectors.detach().numpy().round(3))
    _torch_ok = True
except Exception as e:
    print("torch no disponible, ilustracion con NumPy:", e)
    rng = np.random.default_rng(0)
    E = rng.normal(0, 1, size=(len(vocab), 3))
    print("Tabla de embeddings E:\n", E.round(3))
    print("Busqueda (filas", idx[:5], "):\n", E[idx[:5]].round(3))
    _torch_ok = False

In [ ]:
# Por que los embeddings ganan a one-hot en alta cardinalidad: conteo de parametros/memoria.
K = df["embark_town"].nunique()
for K_demo in [K, 1000, 1_000_000]:
    onehot_dim = K_demo               # ancho de un vector one-hot
    emb_params = K_demo * 8           # una tabla de embeddings de 8-d
    print(f"cardinalidad {K_demo:>9,}: ancho one-hot = {onehot_dim:>9,} (disperso), "
          f"tabla embedding (d=8) = {emb_params:>10,} params (denso, reutilizable)")

## Resumen — chuleta

| Tecnica | Que hace | Formula | Usar cuando | Cuidado con |
|---|---|---|---|---|
| **One-hot** | categoria → vector indicador | $e_c\in\{0,1\}^K$ | categoricas de baja cardinalidad | explota / dispersa con $K$ alto; elimina una para evitar colinealidad |
| **Feature hashing** | categoria → buckets fijos via hash | $\phi(x)_j=\sum_{h(i)=j}\xi(i)x_i$ | cardinalidad alta/desconocida, streaming | colisiones; no interpretable |
| **Binning** | continua → cajones ordinales | ancho uniforme vs bordes por cuantil | umbrales, domesticar sesgo/outliers | el numero de bins es hiperparametro; pierde info |
| **Estandarizacion** | media 0, varianza 1 | $z=\frac{x-\mu}{\sigma}$ | SVM, k-NN, PCA, lineal+reg, NN | ajustar solo en train |
| **Min-max** | escalar a $[0,1]$ | $\frac{x-x_{\min}}{x_{\max}-x_{\min}}$ | entradas acotadas (pixeles) | los outliers estiran el rango |
| **Normalizar L2** | *filas* de longitud 1 | $\mathbf{x}/\lVert\mathbf{x}\rVert_2$ | similitud coseno, TF-IDF | es por muestra, no por feature |
| **PCA** | proyeccion que maximiza varianza | autovec. de $\Sigma=\frac{1}{n-1}X^\top X$ | decorrelacionar, comprimir, visualizar | estandarizar primero; pierde interpretabilidad |
| **Embeddings** | vectores densos aprendidos | $\mathbf{e}_c=E^\top\text{onehot}(c)$ | alta cardinalidad, importa el parecido | requiere datos de entrenamiento; es parte del modelo |

**Reglas de oro**

1. **Ajusta las transformaciones solo en el split de entrenamiento** y luego
   aplicalas a val/test (sin leakage). En produccion, persiste el transformador
   ajustado o, mejor aun, usa un **feature store** (Notebook 2).
2. Empareja la transformacion con el *modelo*: a los ensembles de arboles casi no
   les importa el escalado; a los modelos basados en distancia/gradiente, mucho.
3. Prefiere pipelines (`sklearn.pipeline.Pipeline`, `ColumnTransformer`) para que
   los mismos pasos corran identicos en entrenamiento y en serving.